In [ ]:
%pip install -qU pypdf langchain-community langchain-text-splitters

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# PyPDFLoader를 쓰면 이미지가 load 안됨
pdf_file_path = "./income_tax.pdf"
loader = PyPDFLoader(pdf_file_path)
pages = []
async for page in loader.alazy_load():
    pages.append(page)

In [ ]:
# 질문에 대한 답변 부분이 나오지 않는다 (pdf에서 이미지로 되어있기 때문에 로딩이 안됨)
pages[34]

In [ ]:
%pip install zerox
%pip install py-zerox

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
from dotenv import load_dotenv
from langchain_community.document_loaders.pdf import ZeroxPDFLoader

load_dotenv()

loader = ZeroxPDFLoader(
    file_path="./income_tax.pdf",
    model="gpt-4.1-mini",
)

In [ ]:
# pages 변수 안에 Document 객체로 저장된 markdown 문서가 있음
pages = loader.load()

In [ ]:
# 137
len(pages)

In [ ]:
# answer target check
pages[34]

In [ ]:
# 문서가 잘 load 됐는지 확인하는 코드

# empty_pages = [p.metadata["page"] for p in pages if p.page_content == ""]
# print(f"빈 페이지: {len(empty_pages)}개 / 전체: {len(pages)}개")

In [ ]:
# create income_tax.txt file

with open("income_tax.txt", "w") as f:
    for page in pages:
        f.write(page.page_content + "\n\n")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, chunk_overlap=100, separators=["\n\n", "\n"]
)

In [ ]:
from langchain_community.document_loaders import TextLoader

# 생성된 incom_tax.txt를 loading
text_path = "./income_tax.txt"
loader = TextLoader(text_path)
document_list = loader.load_and_split(text_splitter)

In [ ]:
document_list[60]

In [ ]:
# Vector DB
%pip install -q langchain-chroma

In [ ]:
# Embedding
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
from langchain_chroma import Chroma

# 기존 db 불러오기
# vector_store = Chroma(
#     collection_name="income_tax_collection",
#     embedding_function=embeddings,
#     persist_directory="./income_tax_collection",
# )


# create vecvorDB
vector_store = Chroma.from_documents(
    documents=document_list,
    embedding=embeddings,
    collection_name="income_tax_collection",
    persist_directory="./income_tax_collection",
)

In [ ]:
# 문서를 3개정도 가져와보는 retriever 생성
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [ ]:
# test query
query = "연봉 5천만원인 직장인의 소득세는?"

In [ ]:
# test retrieval
retriever.invoke(query)

문서 split & load와 vector db embedding, retrieval 세팅이 위에 끝났으니,
아래에는 이제 Graph를 정의하고 구성 (Node, State, Edge)

In [ ]:
from typing_extensions import List, TypedDict
from langchain_core.documents import Document


# create state
class AgentState(TypedDict):
    query: str
    context: List[Document]
    answer: str

In [ ]:
from langgraph.graph import StateGraph

# 방금 정의한 AgentState 구조를 가진 Graph 틀을 생성
graph_builder = StateGraph(AgentState)

In [ ]:
def retrieve(state: AgentState):
    query = state["query"]
    docs = retriever.invoke(query)
    return {"context": docs}

In [ ]:
%pip install -q langchain langchain-openai

In [ ]:
from langsmith import Client
from langchain_openai import ChatOpenAI

# 실제 답변을 생성할 때 쓸 prompt와 llm model 선택
client = Client()
prompt = client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)

llm = ChatOpenAI(model="gpt-4o")

In [ ]:
# create generate node
def generate(state: AgentState):
    context = state["context"]
    query = state["query"]
    rag_chain = prompt | llm
    response = rag_chain.invoke({"question": query, "context": context})
    return {"answer": response}

In [ ]:
# 위에 생성한 node 들을 graph에 node로 등록
graph_builder.add_node("retrieve", retrieve)
graph_builder.add_node("generate", generate)

In [ ]:
# graph node들을 edge로 연결
from langgraph.graph import START, END

graph_builder.add_edge(START, "retrieve")
graph_builder.add_edge("retrieve", "generate")
graph_builder.add_edge("generate", END)

In [ ]:
# compile graph
graph = graph_builder.compile()

In [ ]:
# graph 그림으로 그리기
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# test
initial_state = {"query": query}
graph.invoke(initial_state)